# ROGII Particle Filter Lab: Anchor Blending

This notebook is an independently implemented research sandbox for one common geosteering idea: track the structural state `U = TVT + Z` with particles, score candidate TVT positions against the typewell GR curve, and blend the result with the last-visible-TVT anchor.

The goal is not to present a finished leaderboard model. It is to make the state transition, GR calibration, multi-seed weighting, tail failures, and anchor blend visible on a deterministic training-well sample. The notebook creates no `submission.csv`, uses no public prediction artifact, and spends no competition submission.

Why blend? A particle tracker can correct large anchor drift when GR alignment is informative, but an ambiguous GR signature can select the wrong branch. Pooled RMSE rewards the posterior mean, while a conservative anchor component limits catastrophic branch errors.

In [ ]:
from pathlib import Path
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
TRAIN_DIR = DATA_ROOT / 'train'
SAMPLE_SIZE = 30
SAMPLE_SEED = 20260717
N_PARTICLES = 300
N_SEEDS = 8
SEED_TEMPERATURE = 20.0
all_paths = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
paths = sorted(np.random.default_rng(SAMPLE_SEED).choice(all_paths, SAMPLE_SIZE, replace=False))
print('Data root:', DATA_ROOT)
print('Deterministic sample:', len(paths), 'of', len(all_paths), 'wells')

## Model in plain language

1. The visible prefix supplies an exact final TVT anchor and a robust initial rate for `U = TVT + Z`.
2. Typewell GR is mapped to horizontal-well GR with a robust affine calibration fitted only on visible prefix pairs.
3. Each particle carries `(U, rate)`. Rate follows a slowly varying process; the observed trajectory `Z` converts each particle back to TVT for GR likelihood evaluation.
4. Particles are resampled when effective sample size falls below half the population.
5. Eight independent seeds are combined by tempered sequence evidence. A temperature of 20 keeps evidence useful without letting one stochastic path completely dominate.
6. We report the raw particle mean and conservative fixed blends with the anchor.

All parameters below are exposed. They are research defaults, not leaderboard-tuned claims.

In [ ]:
def boundary(frame):
    observed = frame.TVT_input.notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] < 2 or observed[missing[0]:].any():
        raise ValueError('Expected a visible TVT prefix and contiguous missing suffix')
    return int(missing[0])

def calibrate_gr(expected, observed):
    mask = np.isfinite(expected) & np.isfinite(observed)
    x, y = expected[mask], observed[mask]
    if len(x) < 20:
        return 1.0, 0.0, 30.0
    keep = np.ones(len(x), dtype=bool)
    gain, offset = 1.0, 0.0
    for _ in range(3):
        design = np.column_stack((x[keep], np.ones(keep.sum())))
        gain, offset = np.linalg.lstsq(design, y[keep], rcond=None)[0]
        gain = float(np.clip(gain, 0.35, 2.5))
        offset = float(np.median(y[keep] - gain*x[keep]))
        residual = y - (gain*x + offset)
        center = np.median(residual[keep])
        mad = 1.4826*np.median(np.abs(residual[keep]-center))
        new_keep = np.abs(residual-center) <= 3.5*max(mad, 1e-8)
        if new_keep.sum() < 20 or np.array_equal(new_keep, keep):
            break
        keep = new_keep
    residual = y - (gain*x + offset)
    sigma = 1.4826*np.median(np.abs(residual-np.median(residual)))
    return gain, offset, float(np.clip(sigma, 10.0, 60.0))

def initial_rate(md, z, tvt, window=30):
    md, z, tvt = md[-window:], z[-window:], tvt[-window:]
    dm = np.diff(md)
    valid = dm > 0
    if valid.sum() < 3:
        return 0.0
    rate = np.diff(tvt+z)[valid] / dm[valid]
    return float(np.clip(np.median(rate), -0.25, 0.25))

def systematic_resample(weights, rng):
    cumulative = np.cumsum(weights)
    cumulative[-1] = 1.0
    points = rng.uniform(0, 1/len(weights)) + np.arange(len(weights))/len(weights)
    return np.searchsorted(cumulative, points, side='left')

In [ ]:
def run_seed(seed, md, z, gr, previous_md, anchor_u, rate0, tw_tvt, expected_hgr, gr_sigma):
    rng = np.random.default_rng(seed)
    u = anchor_u + 4.5*rng.standard_normal(N_PARTICLES)
    rate = rate0 + 0.01*rng.standard_normal(N_PARTICLES)
    weights = np.full(N_PARTICLES, 1/N_PARTICLES)
    prediction = np.empty(len(md))
    log_evidence = 0.0
    for row in range(len(md)):
        dm = max(float(md[row]-previous_md), 1.0)
        rate = 0.998*rate + 0.002*rng.standard_normal(N_PARTICLES)
        u = u + rate*dm + 0.005*rng.standard_normal(N_PARTICLES)
        particle_tvt = np.clip(u-z[row], tw_tvt[0]-100, tw_tvt[-1]+100)
        u = particle_tvt + z[row]
        if np.isfinite(gr[row]):
            expected = np.interp(particle_tvt, tw_tvt, expected_hgr)
            scaled = (gr[row]-expected)/gr_sigma
            likelihood = np.exp(np.clip(-0.5*scaled*scaled, -50, 0))
            evidence = float(weights@likelihood)
            log_evidence += np.log(max(evidence, 1e-300))
            weights *= likelihood
            weights /= max(weights.sum(), 1e-300)
        effective_n = 1.0/float(weights@weights)
        if effective_n < 0.5*N_PARTICLES:
            selected = systematic_resample(weights, rng)
            u = u[selected] + 0.10*rng.standard_normal(N_PARTICLES)
            rate = rate[selected] + 0.001*rng.standard_normal(N_PARTICLES)
            weights.fill(1/N_PARTICLES)
        prediction[row] = float(weights@(u-z[row]))
        previous_md = md[row]
    return prediction, log_evidence

def particle_ensemble(frame, typewell, seed_base):
    b = boundary(frame)
    md = frame.MD.to_numpy(float); z = frame.Z.to_numpy(float)
    gr = frame.GR.to_numpy(float); known = frame.TVT_input.to_numpy(float)[:b]
    tw = typewell[['TVT','GR']].dropna().sort_values('TVT').drop_duplicates('TVT')
    tw_tvt = tw.TVT.to_numpy(float); tw_gr = tw.GR.to_numpy(float)
    gain, offset, gr_sigma = calibrate_gr(np.interp(known, tw_tvt, tw_gr), gr[:b])
    expected_hgr = gain*tw_gr + offset
    rate0 = initial_rate(md[:b], z[:b], known)
    paths, evidence = [], []
    for seed in range(seed_base, seed_base+N_SEEDS):
        pred, log_ev = run_seed(seed, md[b:], z[b:], gr[b:], md[b-1], known[-1]+z[b-1], rate0, tw_tvt, expected_hgr, gr_sigma)
        paths.append(pred); evidence.append(log_ev)
    evidence = np.asarray(evidence)
    logits = (evidence-evidence.max())/SEED_TEMPERATURE
    seed_weights = np.exp(np.clip(logits, -50, 0)); seed_weights /= seed_weights.sum()
    prediction = np.average(np.stack(paths), axis=0, weights=seed_weights)
    diagnostics = {
        'gr_sigma': gr_sigma, 'initial_rate': rate0,
        'effective_seeds': 1/float(seed_weights@seed_weights),
        'best_seed_weight': float(seed_weights.max()),
    }
    return b, prediction, diagnostics

## Deterministic true-start backtest

The sample is selected once from sorted well paths with a fixed seed. Every candidate sees only `TVT_input` before the supplied missing boundary; full `TVT` is used only for scoring. We accumulate row-level squared error before taking RMSE and also retain per-well errors for tail analysis.

In [ ]:
records = []
started = time.perf_counter()
for number, path in enumerate(paths, 1):
    well_id = path.name.removesuffix('__horizontal_well.csv')
    frame = pd.read_csv(path)
    typewell = pd.read_csv(path.with_name(f'{well_id}__typewell.csv'))
    b, particle, diag = particle_ensemble(frame, typewell, SAMPLE_SEED+1000*number)
    truth = frame.TVT.to_numpy(float)[b:]
    anchor = np.full(len(truth), float(frame.TVT_input.iloc[b-1]))
    candidates = {
        'anchor': anchor,
        'particle': particle,
        'blend50': 0.50*anchor + 0.50*particle,
        'blend65': 0.35*anchor + 0.65*particle,
        'blend75': 0.25*anchor + 0.75*particle,
    }
    record = {'well_id': well_id, 'n_eval': len(truth), **diag}
    for name, pred in candidates.items():
        error = pred-truth
        record[f'sse_{name}'] = float(error@error)
        record[f'rmse_{name}'] = math.sqrt(record[f'sse_{name}']/len(error))
    records.append(record)
    print(f"{number:02d}/{len(paths)} {well_id} anchor={record['rmse_anchor']:.2f} particle={record['rmse_particle']:.2f}")
results = pd.DataFrame(records)
print('Runtime seconds:', round(time.perf_counter()-started, 1))

In [ ]:
rows = []
for name in ['anchor','particle','blend50','blend65','blend75']:
    per_well = results[f'rmse_{name}']
    rows.append({
        'candidate': name,
        'pooled_rmse': math.sqrt(results[f'sse_{name}'].sum()/results.n_eval.sum()),
        'macro_rmse': per_well.mean(),
        'median': per_well.median(),
        'p90': per_well.quantile(0.90),
        'p95': per_well.quantile(0.95),
        'worst': per_well.max(),
    })
summary = pd.DataFrame(rows).set_index('candidate').sort_values('pooled_rmse')
display(summary.style.format('{:.3f}'))

In [ ]:
results['particle_minus_anchor'] = results.rmse_particle-results.rmse_anchor
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(results.rmse_anchor, results.rmse_particle, c=results.best_seed_weight, cmap='viridis', s=35)
limit = max(results.rmse_anchor.max(), results.rmse_particle.max())
axes[0].plot([0,limit],[0,limit],'k--',linewidth=1)
axes[0].set(title='Per-well particle versus anchor', xlabel='anchor RMSE', ylabel='particle RMSE')
axes[1].scatter(results.effective_seeds, results.particle_minus_anchor, c=results.gr_sigma, cmap='plasma', s=35)
axes[1].axhline(0,color='black',linewidth=1)
axes[1].set(title='Seed diversity is not a complete failure gate', xlabel='effective seed count', ylabel='particle minus anchor RMSE')
plt.tight_layout(); plt.show()
display(results.nlargest(10,'particle_minus_anchor')[[
    'well_id','n_eval','rmse_anchor','rmse_particle','rmse_blend65',
    'gr_sigma','initial_rate','effective_seeds','best_seed_weight'
]])

## What I would require before submission

A favorable pooled score on 30 wells is only a smoke test. Before using a daily submission, I would require:

- at least 100 deterministic wells, followed by a full 773-well confirmation;
- stability across particle seeds, particle counts, and evidence temperatures;
- p95/worst-well improvement, not only pooled improvement;
- a prefix-only failure gate validated out of well, if diagnostics can actually separate wrong branches;
- a notebook rerun that is deterministic, internet-off, and comfortably inside the nine-hour limit.

The main reusable idea is not a particular constant. It is the candidate architecture: **physics-aware tracker plus an anchor safety prior, evaluated at the true supplied boundary with explicit tail reporting**.